# 002 SOT Modeling Credit Card Fraud

## Header

In [ ]:
from notebook_config import setup

In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import settings

from src.ingestion.upload_dataset import SOR_BUCKET_KEY
from src.utils.s3_client import get_s3_client

In [ ]:
import logging
logger = logging.getLogger(__name__)

## Dataset: Credit Card Fraud

In [ ]:
bucket = settings.MINIO_BUCKET
key = SOR_BUCKET_KEY

s3 = get_s3_client()
obj = s3.get_object(Bucket=bucket, Key=SOR_BUCKET_KEY)
payload = obj['Body'].read()

df = pd.read_parquet(io.BytesIO(payload))
df.head()

Definindo contrato de transformação SOR -> SOT.

In [ ]:
df.columns

In [ ]:
EXPECTED_COLUMNS = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
ALLOWED_CLASSES = {0, 1}

In [ ]:
def validate_input(df):
    logger.info("Validando dados SOR")

    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]

    assert not missing_cols, f"Colunas faltando: {missing_cols}"
    assert not extra_cols, f"Colunas extras: {extra_cols}"
    logger.info("Colunas validadas")

    assert pd.api.types.is_numeric_dtype(df["Time"]), "Time deve ser numérico"
    assert pd.api.types.is_numeric_dtype(df["Amount"]), "Amount deve ser numérico"
    assert pd.api.types.is_numeric_dtype(df["Class"]), "Class deve ser numérico"
    logger.info("Tipos das Colunas validados")

    assert (df["Amount"] >= 0).all(), "Amount contém valores negativos"
    assert set(df["Class"].dropna().unique()).issubset(ALLOWED_CLASSES), "Class fora de {0,1}"
    logger.info("Valores das Colunas validados")

In [ ]:
validate_input(df)

In [ ]:
def build_sot(df) -> pd.DataFrame:

    sot_df: pd.DataFrame = df.copy()

    sot_df["transaction_id"] = pd.util.hash_pandas_object(
        sot_df[EXPECTED_COLUMNS], index=False
    ).astype("uint64").astype(str)
    logger.info("Coluna transaction_id criada")


    sot_df["amount_log"] = np.log1p(sot_df["Amount"]).astype("float64")
    logger.info("Coluna amount_log criada")

    sot_df["Time"] = sot_df["Time"].astype("int64")
    logger.info("Coluna Class tipada em int64")
    sot_df["Class"] = sot_df["Class"].astype("int8")
    logger.info("Coluna Class tipada em int8")
    sot_df["Amount"] = sot_df["Amount"].astype("float64")
    logger.info("Coluna Amount tipada em float64")

    sot_df = sot_df.sort_values("Time").reset_index(drop=True)
    logger.info("DataFrame ordenado por Time")

    return sot_df

In [ ]:
sot_df = build_sot(df)

In [ ]:
sot_df